In [0]:
-- ============================================================
-- 1. INITIAL DATA EXPLORATION
-- ============================================================
SELECT *
FROM workspace.default.retail_sales_dataset
LIMIT 50;

-- ============================================================
-- 2. COLUMN RENAMING (standardize to lowercase snake_case)
-- ============================================================
ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Transaction_ID` TO transaction_id;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Transaction_Date` TO date;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Customer_ID` TO customer_id;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Gender` TO gender;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Age` TO age;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Product_Category` TO product_category;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Quantity` TO quantity;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Price_per_Unit` TO price_per_unit;

ALTER TABLE workspace.default.retail_sales_dataset
  RENAME COLUMN `Total_Amount` TO total_amount;

-- ============================================================
-- 3. DATA QUALITY CHECKS
-- ============================================================

-- Preview after renaming
SELECT *
FROM workspace.default.retail_sales_dataset
LIMIT 10;

-- Check for duplicate rows
SELECT (*),
       COUNT(*)
FROM workspace.default.retail_sales_dataset
GROUP BY ALL;

-- Check for NULL values across all columns
SELECT *
FROM workspace.default.retail_sales_dataset
WHERE transaction_id IS NULL
   OR transaction_date IS NULL
   OR customer_id IS NULL
   OR gender IS NULL
   OR age IS NULL
   OR quantity IS NULL
   OR price_per_unit IS NULL
   OR product_category IS NULL
   OR total_amount IS NULL;
-- No null values found

-- Date range check
SELECT MIN(transaction_date),
       MAX(transaction_date)
FROM workspace.default.retail_sales_dataset;

-- ============================================================
-- 4. EXPLORATORY BUSINESS ANALYSIS
-- ============================================================

-- 4.1 Total revenue
SELECT SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset;

-- 4.2 Total revenue per product category
SELECT Product_category,
       SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset
GROUP BY 1;

-- 4.3 Total revenue per product category and gender
SELECT gender,
       Product_category,
       SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset
GROUP BY 1, 2;

-- 4.4 Total revenue per product category (alternate grouping)
SELECT product_category,
       SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset
GROUP BY ALL;

-- 4.5 Total revenue by month name
SELECT MONTHNAME(transaction_date) AS month_name,
       SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset
GROUP BY ALL;

-- 4.6 Total revenue by age group
-- Age bands: 15-30 Youth | 31-45 Adults | 46-55 Senior | 56+ Retired
SELECT MONTHNAME(transaction_date) AS month_name,
       Age,
       CASE
           WHEN age BETWEEN '15' AND '30' THEN 'Youth'
           WHEN age BETWEEN '31' AND '45' THEN 'Adults'
           WHEN age BETWEEN '46' AND '55' THEN 'Senior'
           ELSE 'Retired'
       END AS Age_Groups,
       Product_category,
       SUM(total_amount) AS Total_revenue
FROM workspace.default.retail_sales_dataset
GROUP BY ALL;

-- 4.7 Customer count per age group
SELECT
    CASE
        WHEN Age BETWEEN 15 AND 30 THEN 'Youth'
        WHEN Age BETWEEN 31 AND 45 THEN 'Adults'
        WHEN Age BETWEEN 46 AND 55 THEN 'Senior'
        ELSE 'Retired'
    END AS Age_Groups,
    COUNT(*) AS Total_Customers
FROM workspace.default.retail_sales_dataset
GROUP BY
    CASE
        WHEN Age BETWEEN 15 AND 30 THEN 'Youth'
        WHEN Age BETWEEN 31 AND 45 THEN 'Adults'
        WHEN Age BETWEEN 46 AND 55 THEN 'Senior'
        ELSE 'Retired'
    END;

-- 4.8 Date extraction helpers (month name, month id, day name, day number)
SELECT transaction_date,
       MONTHNAME(transaction_date) AS month_name,
       DATE_FORMAT(transaction_date, 'yyyy-MMM') AS month_id,
       DAYNAME(transaction_date) AS day_name,
       DAYOFWEEK(transaction_date) AS day_number
FROM workspace.default.retail_sales_dataset;

-- 4.9 One-time customers by product category
SELECT
    `Product_Category`,
    COUNT(DISTINCT `Customer_ID`) AS one_time_customers,
    SUM(`Total_Amount`) AS total_revenue,
    ROUND(AVG(`Total_Amount`), 2) AS avg_transaction_value,
    ROUND(COUNT(DISTINCT `Customer_ID`) * 100.0 / 1000, 1) AS pct_of_one_time_customers
FROM
    `workspace`.`default`.`retail_sales_dataset`
WHERE
    `Customer_ID` IS NOT NULL
GROUP BY
    `Product_Category`
ORDER BY one_time_customers DESC;

-- 4.10 Customers with exactly one transaction (sample)
SELECT Customer_ID, COUNT(*) AS txn_count
FROM workspace.default.retail_sales_dataset
GROUP BY Customer_ID
HAVING COUNT(*) = 1
LIMIT 5;

-- ============================================================
-- 5. VERIFICATION OF RETAIL INSIGHTS AGENT ANSWERS
-- ============================================================

-- 5.1 Is there a noticeable difference in spending between
--     male and female customers?
SELECT gender,
       SUM(total_amount) AS Spendings
FROM workspace.default.retail_sales_dataset
GROUP BY gender;

-- Ground truth ordering for "Top 5 customers by total spending" reference
SELECT Gender, ROUND(SUM(`Total_Amount`), 2) AS total_spend
FROM workspace.default.retail_sales_dataset
GROUP BY Gender
ORDER BY total_spend;

-- 5.2 Which product category generates the highest total revenue
--     and which one generates the lowest sales?
SELECT Product_category,
       SUM(total_amount) AS Spendings,
       ROUND(SUM(total_amount) * 100.0 / SUM(SUM(total_amount)) OVER (), 2) AS Percentage
FROM workspace.default.retail_sales_dataset
GROUP BY Product_category
ORDER BY SUM(total_amount) DESC;

-- Same result, alternate query form
SELECT
    `Product_Category`,
    SUM(`Total_Amount`) AS total_revenue
FROM
    `workspace`.`default`.`retail_sales_dataset`
WHERE
    `Product_Category` IS NOT NULL
GROUP BY
    `Product_Category`
ORDER BY
    total_revenue DESC;

-- 5.3 How does average daily revenue change across the week?

-- v1: total spend per day-name, with % share (sum-based, not the
--     validated metric — kept for reference)
SELECT
    DAYNAME(transaction_date) AS Day,
    SUM(total_amount) AS Spendings,
    AVG(total_amount) AS Average_Revenue,
    ROUND(SUM(total_amount) * 100.0 / SUM(SUM(total_amount)) OVER (), 2) AS Percentage
FROM workspace.default.retail_sales_dataset
GROUP BY DAYNAME(transaction_date)
ORDER BY SUM(total_amount) DESC;

-- v2: true average daily revenue per day of week
--     (SUM / COUNT DISTINCT transaction_date) — this is the corrected
--     query used for the final validation
SELECT
    DAYOFWEEK(`Transaction_Date`) AS day_num,
    CASE DAYOFWEEK(`Transaction_Date`)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_of_week,
    SUM(`total_amount`) / COUNT(DISTINCT `Transaction_Date`) AS avg_daily_revenue
FROM
    workspace.default.retail_sales_dataset
WHERE
    `Transaction_Date` IS NOT NULL
GROUP BY
    DAYOFWEEK(`Transaction_Date`),
    day_of_week
ORDER BY
    day_num;

-- v3 (also used earlier): DAYOFWEEK-based average, no CASE labeling
SELECT
    DAYOFWEEK(`Transaction_Date`) as day_num,
    SUM(`total_amount`) / COUNT(DISTINCT `Transaction_Date`) AS avg_daily_revenue
FROM
    workspace.default.retail_sales_dataset
WHERE
    `Transaction_Date` IS NOT NULL
GROUP BY
    DAYOFWEEK(`Transaction_Date`)
ORDER BY
    day_num;

-- 5.4 What was the profit margin last quarter?
--     (No cost/COGS field exists in this dataset — this query returns
--     revenue by quarter, used to confirm the agent's fallback figures,
--     not an actual profit margin)
SELECT COUNT(transaction_id) AS Total_transactions,
       QUARTER(transaction_date) AS year_quarter,
       SUM(total_amount) AS Spendings
FROM workspace.default.retail_sales_dataset
GROUP BY QUARTER(transaction_date)
ORDER BY QUARTER(transaction_date) DESC;